<a href="https://colab.research.google.com/github/supsi-dacd-isaac/TeachDecisionMakingUncertainty/blob/main/L13/decision_under_uncertainty_multistage_deepc_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model predictive control for peak shaving
In this exercise we'll see how to apply one flavor of **multistage stochastic control** exploiting probabilistic forecasting. More specifically, we will use model predictive control to continuously optimize the charging and discharging operations of an electric battery. 

In our toy example we'll use [open data of the total electric consumption of the city of Zurich](https://opendata.swiss/en/dataset/viertelstundenwerte-zur-stromabgabe-in-den-netzebenen-5-und-7-in-der-stadt-zurich-seit-2010/resource/371ea4c4-7b78-47ee-a821-f7e1512b2b5f) as power profile on which performing peak shaving.   

In [ ]:
import json
import cvxpy as cvx
import numpy as np
import pandas as pd
from os.path import join
import matplotlib.pyplot as plt
import networkx as nx
from copy import deepcopy
from tqdm import tqdm
np.random.seed(100)

We start by defining some parameters of the electric battery we are going to optimize. Here $x_{max}$ and $x_{min}$ represent the maximum and minimum energy that can be stored in the battery. Units are meant in MWh. $p_{max}$, $\eta$ and $\tau_{self discharge}$ are the maximum charging/discharging power in MW, the (symmetric) efficiency and the self-discharge time constant of the battery.  



In [ ]:
# battery pars
pars = {'x_max':24, 
        'x_min':1,
        'p_max':24,
        'eta':0.98,
        'tau_self_discharge': 1/365/24/3600}

# add dynamic matrices
pars.update({'A': 1-pars['tau_self_discharge'],
             'B':np.array([pars['eta'], -1/pars['eta']]).reshape(1, -1)})

[print('{}: {}'.format(k, v)) for k,v in pars.items()];

## Stochastic MPC for peak shaving
To model the optimization problem we'll use `cvxpy`, a high-level optimization language for [disciplined convex programs](https://www.cvxpy.org/tutorial/dcp/index.html) for python. We would like to perform peak shaving, penalizing more high peaks. The mathematical formulation of the problem can be written as:
$$
\begin{align}
\min_{p^{ch}, p^{di}}& \ \operatorname{CVaR}_{1-\alpha}\left[\sum_{t=1}^T (\hat{p}_{t, s} + p_{t, s}^{ch}-p_{t, s}^{di})^2\right]\\
s.t.: & x_{t+1, s} = A x_{t, s} + B [p_{t, s}^{ch}, p_{t, s}^{di}]^T \quad \forall t, s\\
     & p^{ch}, p^{di} \in \mathcal{P}, \ x \in \mathcal{X} 
\end{align}
$$   

where $x$ is the energy of the battery, $\mathcal{P}$ and $\mathcal{X}$ are (rectangular) set of constraints on the power and the state of the battery, respectively. Here $\hat{p}_{t, s}$ stands for the forecasted power of the municipality of Zurich at the $t_{th}$ step ahead in the $s_{th}$ forecasted scenario. This implies that both the inputs ($\hat{p}$) and the output ($p_{t, s}^{ch}, p_{t, s}^{di}$) of the optimization can be seen as matrices with a temporal and a scenario dimension. 

As we recall from the lesson, the CVaR is the integral value over the 
The $\text{CVaR}_(1-\alpha)$ operator is defined as the integral of the loss function above the $\alpha$ quantile:
$$\operatorname{CVaR}_{1-\alpha}(w)=\frac{1}{\alpha} \int_{F_W(z) \geqslant 1-\alpha} zdz$$
Since we are using scenarios to approximate the probability distribution of the forecasted quantity, this can be obtained as the sum of the greatest $1-\alpha$ fraction of the loss function across across the scenarios. 

### ❓ Complete the control problem
In the following cell there's the cvxpy code for the above control problem. What's missing are the power and state constraints and the objective function. For writing the CVaR operator, try to use the `cvx.sum_largest` function 

In [ ]:
t = 24 # timesteps
n_scenarios = 10 # number of scenarios

def cast_problem(t, n_scenarios):
  p_ch = cvx.Variable((t, n_scenarios))
  p_di = cvx.Variable((t, n_scenarios))

  x = cvx.Variable((t, n_scenarios))

  # parameters
  x_start = cvx.Parameter((1,))
  p_hat = cvx.Parameter((t, n_scenarios))

  # state constraints
  c = []
  c.append()
  c.append()

  # control constraints
  c.append()
  c.append()
  c.append()
  c.append()

  # dynamic equations
  for s in range(n_scenarios):
    x_scen = cvx.hstack([x_start, x[:, s]])
    u = cvx.hstack([p_ch[:, [s]], p_di[:, [s]]])
    c.append(x_scen[1:] == x_scen[:-1] * pars['A']  + cvx.reshape(u @ pars['B'].T, t))

  obj = 0 # replace this with the CVaR of the sum of squares 
  problem = cvx.Problem(cvx.Minimize(obj), constraints=c)
  return problem, x_start, p_hat, p_ch, p_di, x, c, obj

Let's solve one instance of the problem using the following signal as an example for which is easy to see if we get the indended behavior:
$$\hat{p} = sin(3 t \pi / T) + 0.05\mathcal{w}_t$$
where $t$ is the timestep and $\mathcal{w}_t$ is a random walk process. Since our objective function penalizes more high deviations from the zero signal, we expect to see plateaus in the optimized profile. 

In [ ]:
problem, x_start, p_hat, p_ch, p_di, x, c, obj = cast_problem(t, n_scenarios)
power_profile = np.outer(np.sin(np.arange(t)*3*np.pi/t).reshape(-1, 1), np.ones(n_scenarios)) + 0.05*np.cumsum(np.random.randn(t, n_scenarios), axis=0)
x_start.value = [5]
p_hat.value = power_profile
problem.solve(solver = cvx.CLARABEL)

In [ ]:
def plot_results(x, p_batt, p_hat, ax=None):
  if ax is None:
    fig, ax = plt.subplots(2, 1, figsize=(10, 6))
  l1 = ax[0].plot(x, color='green', alpha=0.4)
  l2 = ax[0].plot(p_batt, color='r', alpha=0.4)
  l3 = ax[1].plot(p_hat, color='violet', alpha=0.4)
  l4 = ax[1].plot(p_hat + p_batt, color='orange', alpha=0.4)
  ax[0].legend([l1[0], l2[0]], ['x', r'$p_{batt}$'],loc='lower right')
  ax[1].legend([l3[0], l4[0]], [r'$\hat{p}$',r'$p_{opt}$'],loc='lower right')
  return l1, l2, l3, l4

plot_results(x.value, p_ch.value-p_di.value, p_hat.value);

### Non-anticipativity constraint
The above solution seems to be correct. But what action will we apply at the next iteration? We have solved the problem using 10 scenarios, and for each of them we have retrieved a different power profile for the battery. However, we can only apply a solution at the next step - we have only one battery, not 10 hypothetical ones!

To solve this problem we add a non-anticipativity constraint to the problem: we require all the first step actions to be the same:
$$p^{ch}_{0, s} = p^{ch}_{0, z}, \quad p^{di}_{0, s} = p^{di}_{0, z}  \quad \forall s, z$$ 

This is called non-antiipative since this prevent us to use information ahead of time: at the first step we are not aware of which temporal trace (scenario) will become true; our best guess is to take an action accomodating all the possible scenarios. 


In [ ]:
def cast_problem_nonanticipativity(t, n_scenarios):
  problem, x_start, p_hat, p_ch, p_di, x, c, obj = cast_problem(t, n_scenarios)
  for s in range(n_scenarios):
    # non-anticipativity constraint
    c.append(p_ch[0, s] ==  p_ch[0, np.minimum(s+1, n_scenarios-1)])
    c.append(p_di[0, s] ==  p_di[0, np.minimum(s+1, n_scenarios-1)])
  problem = cvx.Problem(cvx.Minimize(obj), constraints=c)
  return problem, x_start, p_hat, p_ch, p_di, x
  
problem, x_start, p_hat, p_ch, p_di, x = cast_problem_nonanticipativity(t, n_scenarios)
x_start.value = [5]
p_hat.value = power_profile
problem.solve(solver=cvx.CLARABEL)
plot_results(x.value, p_ch.value-p_di.value, p_hat.value);

## Peak shaving of the city of Zurich power profile
We'll take data from the municipality of Zurich to perform peak shaving. We will resample the data to 1 hour, for computational reasons.

In [ ]:
data = pd.read_csv('https://data.stadt-zuerich.ch/dataset/ewz_stromabgabe_netzebenen_stadt_zuerich/download/ewz_stromabgabe_netzebenen_stadt_zuerich.csv', index_col=0,nrows=96*365, parse_dates=[0])
print(data.head())
data.index = pd.DatetimeIndex(pd.to_datetime(data.index, utc=True))

In [ ]:
data['Value_NE7'].iloc[:96*7].plot()
data.index = pd.to_datetime(data.index)
data = data.resample('1h').mean()
data['Value_NE7'].iloc[:24*7].plot()

### Forecasting and bootstrapped scenarios
In the following code, we use a simple regressor-based forecaster. The forecaster has also a method, called predict_scenarios-

### ❓Can you tell what (and how) the predic_scenarios method does?

In [ ]:
from lightgbm import LGBMRegressor

def dataframe_for_lightgbm(x):
    """Flatten MultiIndex cols to plain strings for sklearn/LGBM; keeps your original x for indexing."""
    X = x.copy()
    if isinstance(X.columns, pd.MultiIndex):
        X.columns = ['|'.join(map(str, tup)) for tup in X.columns]
    else:
        X.columns = [str(c) for c in X.columns]
    return X

class LGBM:
  """
  A simple regressor-based forecaster
  """
  def __init__(self, pars):
    self.pars = pars
    self.target_cols = None
    self.models = []
    self.error_scenarios = None

  def fit(self, x, y):
    self.target_cols = y.columns
    for c in y.columns:
        xl = dataframe_for_lightgbm(x)
        m = LGBMRegressor(**self.pars).fit(xl, y[c].values.ravel())
        self.models.append(m)
    self.error_scenarios = pd.DataFrame(y.values - self.predict(x), index=x.index)
    return self

  def predict(self, x):
    y_hat = []
    for m, c in zip(self.models, self.target_cols):
        xl = dataframe_for_lightgbm(x)
        y_hat.append(pd.Series(m.predict(xl), index=x.index, name=c))
    return pd.concat(y_hat, axis=1)
  
  def predict_scenarios(self, x, n_scenarios):
    y_hat = self.predict(x)
    err_scens = []
    for i in range(len(x)):
      h_filt = self.error_scenarios.index.hour ==x.iloc[[i]].index.hour[0]
      chosen = np.random.choice(self.error_scenarios.index[h_filt], n_scenarios)
      err_scens.append(self.error_scenarios.loc[chosen, :].T)
    return  np.expand_dims(y_hat.values, 2) + np.rollaxis(np.dstack(err_scens), -1)

def get_hankel(df, embedding=24*4*2):
    dfs = {}
    for col in df.columns:
        df_i = pd.concat([df[col].shift(-i) for i in range(embedding)], axis=1).iloc[:-embedding]
        df_i.columns = [i for i in range(df_i.shape[1])]
        dfs[col] = df_i
    return pd.concat(dfs, axis=1)

lagged_mav = lambda x, k: x.copy().rolling('{}d'.format(k)).mean()


In [ ]:
target = 'Value_NE7'
n_days = 2

# detrended profile, rescaled to MWh and embed it using a 2-days embedding
df_emb = get_hankel(data[[target]]-lagged_mav(data[[target]], 24*7), embedding=n_days*24) / 1e3
x = df_emb.loc[:, df_emb.columns.get_level_values(1) < 24]
y = df_emb.loc[:, df_emb.columns.get_level_values(1) >= 24]
x['hour'] = x.index.hour
x['weekday'] = x.index.weekday

# divide in training and tes sets
df = pd.concat({'x':x, 'y':y}, axis=1)
n_tr = int(len(df)*0.7)
x_tr, x_te = df['x'].iloc[:n_tr, :], df['x'].iloc[n_tr:, :]
y_tr, y_te = df['y'].iloc[:n_tr, :], df['y'].iloc[n_tr:, :]

In [ ]:
x_tr

In [ ]:
problem, x_start, p_hat, p_ch, p_di, x, c, obj = cast_problem(t, n_scenarios)
power_profile = np.outer(np.sin(np.arange(t)*3*np.pi/t).reshape(-1, 1), np.ones(n_scenarios)) + 0.05*np.cumsum(np.random.randn(t, n_scenarios), axis=0)
x_start.value = [5]
p_hat.value = power_profile
problem.solve(solver = cvx.CLARABEL)

In [ ]:
lgb_pars = {'n_estimators': 100, 'learning_rate': 0.1, 'num_leaves': 31, 'max_depth': 5}
lgbm = LGBM(lgb_pars).fit(x_tr, y_tr)

In [ ]:

from matplotlib import animation
from IPython.display import HTML
def scen_animation(y_te, y_hat, qs, n_rows=50):
    fig, ax = plt.subplots(1, layout='constrained');
    t = np.arange(y_hat.shape[1])
    line1, = ax.plot(y_hat[0, :], lw=2);
    line2, = ax.plot(y_te[0, :], lw=2);
    line3 = ax.plot(np.squeeze(qs[0, :, :]), 'r', lw=2, alpha=0.1);
    ax.set_ylim(y_te.min() -np.abs(y_te.min())*0.2, y_te.max()*1.2)
    #ax.set_ylim(-13, 13)
    def animate(i):
        line1.set_data(t, y_te[i, :]);
        line2.set_data(t, y_hat[i, :]);
        [line3[j].set_data(t, qsi) for j, qsi in enumerate(qs[i, :, :].T)];
        return (line1, line2, *line3, )

    def init():
        line1.set_data([], []);
        return (line1,)

    ani = animation.FuncAnimation(fig, animate, init_func=init, frames=n_rows, interval=100, blit=True)
    plt.close('all')
    return HTML(ani.to_jshtml())

def plot_sol(p_opt, p_batt_opt, y_te, steps, label):
    plt.figure(figsize=(10, 4))
    plt.plot(p_batt_opt, label=f"battery power, {label}")
    plt.plot(y_te.iloc[:steps, 0].values, label="uncontrolled load")
    plt.plot(p_opt, label=f"controlled load, {label}")
    plt.title(
        f"{label} closed-loop objective: = "
        f"{np.max(p_opt):0.3e}"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

def battery_animation(p_batt, x_batt, p_hat, n_rows=50):
    fig, ax = plt.subplots(2, 1, layout='constrained');
    t = np.arange(p_batt.shape[1])
    p_opt = p_hat + p_batt
    line1, line2, line3, line4 = plot_results(x_batt[0, :, :], p_batt[0, :, :], p_hat[0, :, :], ax=ax)
    ax[0].set_ylim(p_batt.min() -np.abs(p_batt.min())*0.2, p_batt.max()*2)
    ax[1].set_ylim(p_hat.min() -np.abs(p_hat.min())*0.2, p_hat.max()*1.2)
    def animate(i):
        [line1[j].set_data(t, qsi) for j, qsi in enumerate(p_batt[i, :, :].T)];
        [line2[j].set_data(t, qsi) for j, qsi in enumerate(x_batt[i, :, :].T)]
        [line3[j].set_data(t, qsi) for j, qsi in enumerate(p_hat[i, :, :].T)];
        [line4[j].set_data(t, qsi) for j, qsi in enumerate(p_opt[i, :, :].T)];
        return (*line1, *line2, *line3, *line4, )

    def init():
        [line1[j].set_data([], []) for j, qsi in enumerate(line1)];
        return (*line1,)

    ani = animation.FuncAnimation(fig, animate, init_func=init, frames=n_rows, interval=100, blit=True)
    plt.close('all')
    return HTML(ani.to_jshtml())

In [ ]:
close_loop_steps = 24*2
n_scenarios = 20
scens = lgbm.predict_scenarios(x_te.iloc[:close_loop_steps, :], n_scenarios)
scen_animation(y_te.iloc[:close_loop_steps, :].values, lgbm.predict(x_te.iloc[:close_loop_steps, :]).values, scens, n_rows=close_loop_steps)

### Closed loop simulation
In the following cell, the `solve_close_loop` function solves different instances of the MPC base problem.  
❓ Try to complete the following code, which is running the MPC in closed loop

In [ ]:
def solve_close_loop(n_scenarios = 5):
  scens = lgbm.predict_scenarios(x_te.iloc[:close_loop_steps, :], n_scenarios)
  problem, x_start, p_hat, p_ch, p_di, x = cast_problem_nonanticipativity(t, n_scenarios=n_scenarios)

  x_last = 5
  # optimal actions that have been applied by the MPC after the optimization (one per step)
  p_batt_opt_history, x_opt_history = [], []
  # solutions for the whole horizon (n_scenrios * (control horizon) per step)
  p_batt_sol_history, x_sol_history = [], []
  for i in tqdm(range(close_loop_steps)):
    # COMPLETE HERE  

  # reshaping dimensions
  p_batt_sol_history = np.rollaxis(np.dstack(p_batt_sol_history), -1)
  x_sol_history = np.rollaxis(np.dstack(x_sol_history), -1)
  p_batt_opt_history = np.hstack(p_batt_opt_history)
  x_opt_history = np.hstack(x_opt_history)

  # formatting output
  opt_history = {'p_batt': p_batt_opt_history, 'x':x_opt_history}
  sol_history = {'p_batt': p_batt_sol_history, 'x':x_sol_history}
  return opt_history, sol_history, scens

opt_history, sol_history, scens = solve_close_loop(n_scenarios)

In [ ]:
battery_animation(sol_history['p_batt'], sol_history['x'], scens[:close_loop_steps, :],  n_rows=close_loop_steps)

### Number of scenarios
Try to increase the number of scenarios considered to model uncertainty. How does the objective change?

In [ ]:
close_loop_steps = 24*7
n_scenarios = 2
opt_history_1, sol_history, scens = solve_close_loop(n_scenarios)

p_opt = y_te.iloc[:close_loop_steps, 0].values +  opt_history_1['p_batt']
plot_sol(p_opt, opt_history_1['p_batt'], y_te, close_loop_steps, f"CVX, {n_scenarios} scen.")



## DeePC extension: data-driven battery control

In the previous sections the battery model was written explicitly as

$$
x_{k+1} = A x_k + B u_k,
\qquad 
u_k =
\begin{bmatrix}
p^{ch}_k\\
p^{di}_k
\end{bmatrix}.
$$


In **Data-enabled Predictive Control (DeePC)** we replace this explicit state-space equation with a behavioural constraint built from measured input-output trajectories of the battery.

Here the control input is the battery action $(u_k=[p^{ch}_k,p^{di}_k]^T)$, while the measured output is the battery state of charge $(y_k=x_{k+1})$. From a sufficiently rich input-output experiment, we build block-Hankel matrices:

$$
H_L(u^d), \qquad H_L(y^d).
$$

Then each future input-output trajectory is represented as a linear combination of columns of these Hankel matrices:

$$
\begin{bmatrix}
U_p\\
Y_p\\
U_f\\
Y_f
\end{bmatrix} g_s
=
\begin{bmatrix}
u_{\mathrm{ini}}\\
y_{\mathrm{ini}}\\
u_{f,s}\\
y_{f,s}
\end{bmatrix}.
$$

For the stochastic/scenario setting of this exercise, we use one coefficient vector \(g_s\) for each scenario \(s\), but we impose the same first action across all scenarios, exactly as in the multistage MPC formulation.


In [ ]:

def block_hankel(signal, horizon):
    """
    Construct a block-Hankel matrix from a multivariate time series.

    Parameters
    ----------
    signal : array-like, shape (n_steps, n_features)
        Time series used to build the Hankel matrix.
    horizon : int
        Number of stacked time steps.

    Returns
    -------
    H : np.ndarray, shape (horizon * n_features, n_columns)
        Block-Hankel matrix ordered by time block:
        [signal_0; signal_1; ...; signal_{horizon-1}].
    """
    signal = np.asarray(signal, dtype=float)
    if signal.ndim == 1:
        signal = signal[:, None]

    n_steps, n_features = signal.shape
    n_cols = n_steps - horizon + 1
    if n_cols <= 0:
        raise ValueError("The signal is too short for the requested Hankel horizon.")

    return np.vstack([
        signal[i:i + n_cols].T
        for i in range(horizon)
    ])
    
def simulate_battery_experiment(
    n_steps=250,
    x0=None,
    seed=123,
    excitation="safe_prbs",
    amplitude=0.45,
    center_gain=0.35,
    dither=0.05,
):
    """
    Generate a compact but persistently exciting battery input-output trajectory.

    The purpose is not to imitate the daily load profile. The purpose is to
    excite the battery dynamics while keeping the state away from saturation.

    Output y_k is x_{k+1}, as in the previous DeePC formulation.

    excitation:
        - "safe_prbs": multilevel pseudo-random signal with random holding times.
        - "aperiodic_multisine": smooth non-24-periodic multisine.
        - "hybrid": PRBS + aperiodic multisine + small noise.
    """
    rng = np.random.default_rng(seed)

    x_min = pars["x_min"]
    x_max = pars["x_max"]
    p_max = pars["p_max"]
    eta = pars["eta"]
    A = pars["A"]

    x_mid = 0.5 * (x_min + x_max)

    if x0 is None:
        x0 = x_mid

    u_data = np.zeros((n_steps, 2))
    x_data = np.zeros(n_steps + 1)
    x_data[0] = x0

    # Variables for the PRBS part.
    current_level = 0.0
    next_switch = 0

    # Random phases for the multisine part.
    phases = rng.uniform(0, 2 * np.pi, size=4)

    for k in range(n_steps):

        if excitation in ["safe_prbs", "hybrid"]:
            if k >= next_switch:
                hold = rng.integers(1, 5)
                next_switch = k + hold

                levels = np.array([-1.0, -0.7, -0.4, 0.4, 0.7, 1.0])
                current_level = amplitude * p_max * rng.choice(levels)

            prbs_part = current_level
        else:
            prbs_part = 0.0

        if excitation in ["aperiodic_multisine", "hybrid"]:
            # Frequencies intentionally not restricted to 24-hour harmonics.
            multisine_part = amplitude * p_max * (
                0.30 * np.sin(2 * np.pi * k / 17 + phases[0])
                + 0.25 * np.sin(2 * np.pi * k / 31 + phases[1])
                + 0.20 * np.sin(2 * np.pi * k / 47 + phases[2])
                + 0.15 * np.sin(2 * np.pi * k / 73 + phases[3])
            )
        else:
            multisine_part = 0.0

        noise_part = dither * p_max * rng.uniform(-1.0, 1.0)

        # Small centering feedback to avoid spending too much time at x_min/x_max.
        centering_part = -center_gain * (x_data[k] - x_mid)

        net_power = prbs_part + multisine_part + noise_part + centering_part

        # Convert signed net power into charge/discharge variables.
        # Positive -> charge, negative -> discharge.
        if net_power >= 0:
            p_ch_raw = min(net_power, p_max)
            p_di_raw = 0.0
        else:
            p_ch_raw = 0.0
            p_di_raw = min(-net_power, p_max)

        # Respect battery state limits without hard post-hoc clipping.
        max_charge_allowed = max(
            0.0,
            min(p_max, (x_max - A * x_data[k]) / eta),
        )

        max_discharge_allowed = max(
            0.0,
            min(p_max, (A * x_data[k] - x_min) * eta),
        )

        p_ch = min(p_ch_raw, max_charge_allowed)
        p_di = min(p_di_raw, max_discharge_allowed)

        u_data[k, :] = [p_ch, p_di]

        x_data[k + 1] = A * x_data[k] + eta * p_ch - (1 / eta) * p_di

    y_data = x_data[1:, None]

    return u_data, y_data, x_data


def build_deepc_matrices(u_data, y_data, T_f, T_ini):
    """
    Split input-output Hankel matrices into past and future blocks.
    """
    m = u_data.shape[1]
    p = y_data.shape[1]
    L = T_ini + T_f

    H_u = block_hankel(u_data, L)
    H_y = block_hankel(y_data, L)

    U_p = H_u[:m * T_ini, :]
    U_f = H_u[m * T_ini:, :]

    Y_p = H_y[:p * T_ini, :]
    Y_f = H_y[p * T_ini:, :]

    return U_p, Y_p, U_f, Y_f


def cvx_time_major_input_vector(p_ch_col, p_di_col, horizon):
    """
    Build a flat vector:
    [p_ch(0), p_di(0), p_ch(1), p_di(1), ...]^T

    Shape is (2 * horizon,)
    """
    return cvx.hstack([
        item
        for k in range(horizon)
        for item in (p_ch_col[k], p_di_col[k])
    ])





### Scenario-based DeePC optimization problem

The following function mirrors the previous stochastic MPC problem, but the explicit dynamics are removed.  
Instead of enforcing

$$
x_{k+1,s}=Ax_{k,s}+Bu_{k,s},
$$

we enforce the DeePC behavioural relation through the Hankel matrices.

We add two regularization terms:

$$
\lambda_g \|g_s\|_2^2
$$

to avoid unnecessarily large trajectory coefficients, and

$$
\lambda_y \|\sigma_{y,s}\|_2^2
$$

to softly match the initial measured output history. The slack is useful because real data are noisy and because the initial state may not be represented exactly by a finite dataset.


In [ ]:

def cast_deepc_problem_nonanticipativity(
    T_f,
    n_scenarios,
    U_p,
    Y_p,
    U_f,
    Y_f,
    T_ini,
    alpha=0.1,
    lambda_g=1e-4,
    lambda_y=1e4,
):
    """
    Scenario-based DeePC formulation for the battery peak-shaving problem.

    Parameters
    ----------
    T_f : int
        Prediction/control horizon.
    n_scenarios : int
        Number of load scenarios.
    U_p, Y_p, U_f, Y_f : np.ndarray
        Past/future Hankel blocks.
    T_ini : int
        Number of past input-output samples used to condition DeePC.
    alpha : float
        Tail probability used in the empirical CVaR objective.
        For example alpha=0.1 corresponds to CVaR at level 0.9.
    lambda_g : float
        Ridge regularization on the DeePC coefficient vectors.
    lambda_y : float
        Penalty on the output-initialization slack.

    Returns
    -------
    problem, params, variables : tuple
        A CVXPY problem, parameter dictionary, and variable dictionary.
    """
    m = 2
    p = 1
    n_cols = U_p.shape[1]

    # Decision variables.
    p_ch = cvx.Variable((T_f, n_scenarios))
    p_di = cvx.Variable((T_f, n_scenarios))
    x = cvx.Variable((T_f, n_scenarios))
    g = cvx.Variable((n_cols, n_scenarios))
    sigma_y = cvx.Variable((p * T_ini, n_scenarios))

    # Parameters.
    u_ini = cvx.Parameter(m * T_ini)
    y_ini = cvx.Parameter(p * T_ini)
    p_hat = cvx.Parameter((T_f, n_scenarios))

    constraints = []

    # Battery bounds.
    constraints += [
        x >= pars["x_min"],
        x <= pars["x_max"],
        p_ch >= 0,
        p_ch <= pars["p_max"],
        p_di >= 0,
        p_di <= pars["p_max"],
    ]

    for s in range(n_scenarios):
        u_future_s = cvx_time_major_input_vector(p_ch[:, s], p_di[:, s], T_f)

        # DeePC behavioural constraints.
        constraints += [
            U_p @ g[:, s] == u_ini,
            Y_p @ g[:, s] == y_ini + sigma_y[:, s],
            U_f @ g[:, s] == u_future_s,
            Y_f @ g[:, s] == x[:, s],
        ]

    # Non-anticipativity: the first real action must be unique.
    for s in range(1, n_scenarios):
        constraints += [
            p_ch[0, s] == p_ch[0, 0],
            p_di[0, s] == p_di[0, 0],
        ]

    # Scenario costs for peak shaving.
    scenario_costs = cvx.sum(cvx.square(p_hat + p_ch - p_di), axis=0)

    # Empirical CVaR of the scenario costs.
    beta = cvx.Variable()
    excess = cvx.Variable(n_scenarios)
    constraints += [
        excess >= scenario_costs - beta,
        excess >= 0,
    ]

    cvar = beta + (1 / (alpha * n_scenarios)) * cvx.sum(excess)

    reg = lambda_g * cvx.sum_squares(g) + lambda_y * cvx.sum_squares(sigma_y)

    objective = cvar + reg
    problem = cvx.Problem(cvx.Minimize(objective), constraints)

    params = {
        "u_ini": u_ini,
        "y_ini": y_ini,
        "p_hat": p_hat,
    }

    variables = {
        "p_ch": p_ch,
        "p_di": p_di,
        "x": x,
        "g": g,
        "sigma_y": sigma_y,
        "scenario_costs": scenario_costs,
    }

    return problem, params, variables



### DeePC on the synthetic sinusoidal forecast

This first check uses the same synthetic scenario forecast as the earlier MPC cell.  
The important point is that the optimization no longer uses `pars["A"]` and `pars["B"]` inside the constraints. Those matrices are only used here to generate a didactic dataset. In a real experiment, `u_data` and `y_data` would be measured from the battery.


In [ ]:
def compress_deepc_matrices_svd(
    U_p,
    Y_p,
    U_f,
    Y_f,
    energy_threshold=0.999,
    max_rank=None,
):
    H = np.vstack([U_p, Y_p, U_f, Y_f])

    # Row scaling is important because u and y may have different units.
    row_scale = np.std(H, axis=1, keepdims=True)
    row_scale[row_scale < 1e-8] = 1.0
    Hs = H / row_scale

    U, S, Vt = np.linalg.svd(Hs, full_matrices=False)

    energy = S**2
    cumulative_energy = np.cumsum(energy) / np.sum(energy)

    r = np.searchsorted(cumulative_energy, energy_threshold) + 1

    if max_rank is not None:
        r = min(r, max_rank)

    H_basis_scaled = U[:, :r]

    # Map basis back to original units.
    H_basis = H_basis_scaled * row_scale

    n_Up = U_p.shape[0]
    n_Yp = Y_p.shape[0]
    n_Uf = U_f.shape[0]
    n_Yf = Y_f.shape[0]

    i0 = 0
    i1 = n_Up
    i2 = i1 + n_Yp
    i3 = i2 + n_Uf
    i4 = i3 + n_Yf

    U_p_c = H_basis[i0:i1, :]
    Y_p_c = H_basis[i1:i2, :]
    U_f_c = H_basis[i2:i3, :]
    Y_f_c = H_basis[i3:i4, :]

    info = {
        "original_columns": H.shape[1],
        "compressed_columns": r,
        "energy_kept": cumulative_energy[r - 1],
        "largest_singular_value": S[0],
        "smallest_kept_singular_value": S[r - 1],
    }

    return U_p_c, Y_p_c, U_f_c, Y_f_c, info

In [ ]:

# DeePC hyperparameters.
T_ini_deepc = 4
T_f_deepc = t

# Build a data-driven behavioural representation of the battery.
u_data, y_data, x_data_exp = simulate_battery_experiment(
    n_steps=1000,
    x0=5.0,
    seed=123,
    excitation="hybrid",
    dither=0.1
)

U_p, Y_p, U_f, Y_f = build_deepc_matrices(
    u_data=u_data,
    y_data=y_data,
    T_f=T_f_deepc,
    T_ini=T_ini_deepc,
)

U_p_c, Y_p_c, U_f_c, Y_f_c, info = compress_deepc_matrices_svd(
    U_p,
    Y_p,
    U_f,
    Y_f,
    energy_threshold=0.999,
)

deepc_problem, deepc_params, deepc_vars = cast_deepc_problem_nonanticipativity(
    T_f=T_f_deepc,
    n_scenarios=n_scenarios,
    U_p=U_p_c,
    Y_p=Y_p_c,
    U_f=U_f_c,
    Y_f=Y_f_c,
    T_ini=T_ini_deepc,
    alpha=0.1,
    lambda_g=1e-4,
    lambda_y=1e4,
)

# Initial input-output history.
# At the beginning of the example we assume the battery has been idle
# and has remained close to x_start = 5 MWh for the past T_ini samples.
deepc_params["u_ini"].value = np.zeros(2 * T_ini_deepc)
deepc_params["y_ini"].value = np.ones(T_ini_deepc) * 5.0
deepc_params["p_hat"].value = scens[0, :, :]

deepc_problem.solve(solver=cvx.CLARABEL)
print("DeePC objective:", deepc_problem.value)

p_batt_deepc = deepc_vars["p_ch"].value - deepc_vars["p_di"].value
plot_results(
    deepc_vars["x"].value,
    p_batt_deepc,
    deepc_params["p_hat"].value,
);
plt.suptitle("Scenario-based DeePC solution")
plt.tight_layout()



### Closed-loop DeePC simulation

The function below is the DeePC counterpart of `solve_close_loop`.

At each time step:

1. load scenarios are generated as before;
2. DeePC is conditioned on the most recent input-output history;
3. the first non-anticipative battery action is applied;
4. the battery state is updated;
5. the input-output history is shifted forward.

For speed, the default example uses fewer closed-loop steps than the MPC comparison. Increase `close_loop_steps_deepc` only after checking that the solver is fast enough on your machine.


In [ ]:

def solve_close_loop_deepc(
    n_scenarios=5,
    close_loop_steps_deepc=24 * 7,
    T_ini_deepc=4,
    n_data_deepc=900,
    x0=5.0,
    seed=123,
):
    """
    Closed-loop scenario-based DeePC simulation.
    """
    scens_deepc = lgbm.predict_scenarios(
        x_te.iloc[:close_loop_steps_deepc, :],
        n_scenarios,
    )

    u_data, y_data, _ = simulate_battery_experiment(
        n_steps=n_data_deepc,
        x0=x0,
        seed=seed,
    )

    U_p, Y_p, U_f, Y_f = build_deepc_matrices(
        u_data=u_data,
        y_data=y_data,
        T_f=t,
        T_ini=T_ini_deepc,
    )

    U_p_c, Y_p_c, U_f_c, Y_f_c, info = compress_deepc_matrices_svd(
    U_p,
    Y_p,
    U_f,
    Y_f,
    energy_threshold=0.999,
)

    problem, params, vars_ = cast_deepc_problem_nonanticipativity(
        T_f=t,
        n_scenarios=n_scenarios,
        U_p=U_p_c,
        Y_p=Y_p_c,
        U_f=U_f_c,
        Y_f=Y_f_c,
        T_ini=T_ini_deepc,
        alpha=0.1,
        lambda_g=1e-4,
        lambda_y=1e4,
    )

    x_last = x0

    # Recent measured input-output history.
    # y is x_{k+1}, so initially we use an idle history around x0.
    u_hist = np.zeros((T_ini_deepc, 2))
    y_hist = np.ones((T_ini_deepc, 1)) * x0

    p_batt_opt_history = []
    x_opt_history = []
    p_batt_sol_history = []
    x_sol_history = []

    for i in tqdm(range(close_loop_steps_deepc)):
        params["u_ini"].value = u_hist.reshape(-1)
        params["y_ini"].value = y_hist.reshape(-1)
        params["p_hat"].value = scens_deepc[i, :, :]

        problem.solve(solver=cvx.CLARABEL)

        p_ch_0 = vars_["p_ch"].value[0, 0]
        p_di_0 = vars_["p_di"].value[0, 0]
        u0 = np.array([p_ch_0, p_di_0])

        # Apply the first action to the real/simulated battery.
        x_last = float(
            pars["A"] * x_last
            + pars["eta"] * p_ch_0
            - (1 / pars["eta"]) * p_di_0
        )

        p_batt_opt_history.append(p_ch_0 - p_di_0)
        x_opt_history.append(x_last)
        p_batt_sol_history.append(vars_["p_ch"].value - vars_["p_di"].value)
        x_sol_history.append(vars_["x"].value)

        # Shift the measured DeePC initial trajectory.
        u_hist = np.vstack([u_hist[1:, :], u0.reshape(1, -1)])
        y_hist = np.vstack([y_hist[1:, :], [[x_last]]])

    p_batt_sol_history = np.rollaxis(np.dstack(p_batt_sol_history), -1)
    x_sol_history = np.rollaxis(np.dstack(x_sol_history), -1)

    opt_history = {
        "p_batt": np.asarray(p_batt_opt_history),
        "x": np.asarray(x_opt_history),
    }

    sol_history = {
        "p_batt": p_batt_sol_history,
        "x": x_sol_history,
    }

    return opt_history, sol_history, scens_deepc


# Example run.
# You can increase close_loop_steps_deepc after checking computational cost.
close_loop_steps_deepc = 24 * 7
n_scenarios_deepc = 20

opt_history_deepc, sol_history_deepc, scens_deepc = solve_close_loop_deepc(
    n_scenarios=n_scenarios_deepc,
    close_loop_steps_deepc=close_loop_steps_deepc,
    T_ini_deepc=4,
    n_data_deepc=1000,
)

p_opt_deepc = (
    y_te.iloc[:close_loop_steps_deepc, 0].values
    + opt_history_deepc["p_batt"]
)


In [ ]:
plot_sol(p_opt_deepc, opt_history_deepc["p_batt"], y_te, close_loop_steps_deepc, "DeePC")


### What this DeePC extension illustrates

This section is useful pedagogically because it separates two ideas:

- the **load forecast** still comes from the probabilistic forecaster;
- the **battery dynamics** are no longer imposed through \(A\) and \(B\), but through the input-output behavioural data.

So this is not a replacement for probabilistic forecasting. Rather, it is a replacement for the explicit battery model inside the MPC optimizer.

In this toy example the battery is linear, so DeePC is expected to recover behavior similar to model-based MPC if the Hankel data are sufficiently rich. For nonlinear batteries or other flexible assets, the same direct Hankel span is no longer exact; one would typically need local DeePC, kernel/lifted DeePC, or another nonlinear extension.
